# Suite TSS — TypeScript5 simplified by ECMAScript's semantics

`TypeScript5.Simplifiers.OfAny(term, scope)` simplifies a TypeScript5 term by ECMA-262's semantics, its values and
abstract operations `TypeScript5.Abstract`'s, modeled alike in both implementations: what the scope decides folds into
the literal of its value (a negative number, NaN, Infinity, -0 and `undefined` by the names ECMAScript writes them,
`null` and a regular expression a `constant`), an array or an object of such values into its display, and the rest
stays, the residual, its subterms simplified; a short circuit decides what does not depend on what stays. What has no
rule stays too, named with its class in the evaluation inventory (`residues`): a write, a body of statements, a
coroutine (`d`), a built-in method and a host value (`c`), a type, a pattern, JSX (`b`). What ECMAScript makes an
error still raises, as JavaScript would: TypeError and RangeError, by the same messages. Terms are built through the
dialect's `make`, in a transaction (`term`); `text` gives a residual's source, as the printer writes it.

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.TypeScript import TypeScript5
from mbse.Expressions.TypeScript.Patterns import Expressions as TypeScriptPatterns
from mbse.Expressions.TypeScript.RegExp import Expressions as TypeScriptRegExp
from mbse.Expressions.TypeScript.Types import Expressions as TypeScriptTypes
from mbse.Expressions.TypeScript.TypeScript5 import Abstract as A, Expressions as TS, Simplifiers as TV, Text as TSText
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Framework.Simplifiers import NotEvaluated
from mbse.Schemas.Framework import Builders
from support import raises

store = Builders.Store()
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install,
                TypeScript5.install, TypeScriptTypes.install, TypeScriptPatterns.install, TypeScriptRegExp.install]:
    store.transaction(install)


def people(txn):
    """A person, a reference object, with a list of scores."""
    S = txn.Schemas
    return txn.uid("tss.people/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject})
        .spec([lambda _: _.Person.name("tss.Person").ref(True).properties({
            "name": S.OfStr, "age": S.OfInt, "toString": S.OfStr, "scores": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfInt]))})])]))])


store.transaction(people)


def term(spec):
    """The TypeScript5 term `spec` writes, a native value as its literal, the root of a holder built in a transaction."""
    with store.transaction():
        return F.term(store.Expressions.Framework.Expression(lambda e: e.root(TS.DIALECT.spec(spec))))


def simplified(spec, scope=None):
    """The simplification of the term `spec` writes, in `scope`."""
    return TV.OfAny(term(spec), scope)


def text(spec, scope=None):
    """The source of the residual of the term `spec` writes, simplified in `scope`."""
    residual = simplified(spec, scope).residual
    with store.transaction():
        holder = store.Expressions.Framework.Expression(lambda e: e.root(residual))
    return TSText.ToText(holder)


def value(spec, scope=None):
    """The value the term `spec` writes simplifies to, in `scope`: its literal root's."""
    return simplified(spec, scope).evaluate()


def reasons(spec, scope=None):
    """Why what stayed in the residual stayed: each residue's reason and class."""
    return [(residue.reason, residue.category) for residue in simplified(spec, scope).residues]


def make(kind, attributes, *arguments):
    return TS.DIALECT.make(F.Form(kind, attributes, arguments))


name = lambda n: make("identifier", {"name": n})  # noqa: E731: the builders of the terms the cases write
x, y, c, f = name("x"), name("y"), name("c"), name("f")
binary = lambda op, a, b: make("binaryexpression", {"operator": op}, a, b)  # noqa: E731
unary = lambda op, a: make("unaryexpression", {"operator": op}, a)  # noqa: E731
logical = lambda op, a, b: make("logicalexpression", {"operator": op}, a, b)  # noqa: E731
array = lambda *items: make("arrayexpression", {}, *items)  # noqa: E731
hole = make("elision", {})
spread = lambda a: make("spreadelement", {}, a)  # noqa: E731
prop = lambda key, v, computed=False, shorthand=False: make("property", {  # noqa: E731
    "propertyKind": "init", "computed": computed, "method": False, "shorthand": shorthand}, key, v)
obj = lambda *properties: make("objectexpression", {}, *properties)  # noqa: E731
member = lambda o, p, computed=False, optional=False: make(  # noqa: E731
    "memberexpression", {"computed": computed, "optional": optional}, o, p)
call = lambda callee, *args, optional=False: make("callexpression", {"optional": optional}, callee, *args)  # noqa: E731
new = lambda callee, *args: make("newexpression", {}, callee, *args)  # noqa: E731
chain = lambda e: make("chainexpression", {}, e)  # noqa: E731
arrow = lambda body, *params, is_async=False: make(  # noqa: E731
    "arrowfunctionexpression", {"isAsync": is_async}, None, body, *params)
default = lambda target, v: make("assignmentpattern", {}, target, v)  # noqa: E731
rest = lambda target: make("restelement", {}, target)  # noqa: E731
array_pattern = lambda *elements: make("arraypattern", {}, *elements)  # noqa: E731
object_pattern = lambda *properties: make("objectpattern", {}, *properties)  # noqa: E731
conditional = lambda t, a, b: make("conditionalexpression", {}, t, a, b)  # noqa: E731
sequence = lambda *items: make("sequenceexpression", {}, *items)  # noqa: E731
parenthesized = lambda e: make("parenthesizedexpression", {}, e)  # noqa: E731
null = make("constant", {"singleton": "null"}, None)
regexp = lambda pattern, flags="": make("constant", {"pattern": pattern, **({"flags": flags} if flags else {})}, None)  # noqa: E731
this = make("thisexpression", {})
undefined = name("undefined")


def template(*raws_and_expressions):
    """A template literal's spec: its quasis' raw text, its expressions between them."""
    raws, expressions = raws_and_expressions[::2], raws_and_expressions[1::2]
    return lambda u: u.templateliteral(lambda r: r.quasis([
        (lambda raw, last: lambda q: q.raw(raw).tail(last))(raw, i == len(raws) - 1) for i, raw in enumerate(raws)])
        .expressions([TS.DIALECT.spec(e) for e in expressions]))


def tagged(tag, *raws_and_expressions):
    """A tagged template's spec: its tag, its quasis' raw text and its expressions between them."""
    raws, expressions = raws_and_expressions[::2], raws_and_expressions[1::2]
    return lambda u: u.taggedtemplateexpression(lambda r: r.tag(TS.DIALECT.spec(tag)).quasi(lambda q: q.quasis([
        (lambda raw, last: lambda e: e.raw(raw).tail(last))(raw, i == len(raws) - 1) for i, raw in enumerate(raws)])
        .expressions([TS.DIALECT.spec(e) for e in expressions])))


number_type = lambda t: t.tsnumberkeyword(lambda k: k)  # noqa: E731: a type, `TypeScript Types`' term

<a id="TSS-01"></a>
## TSS-01 · Literals, names and `this`: what the scope binds, the global object's values, and values written back as literals

In [ ]:
# A literal is its value; a name what the scope binds, else the global object's `undefined`, `NaN` and `Infinity`
assert value(1.5) == 1.5 and value(2) == 2 and value("a") == "a" and value(True) is True  # a number is a float, a bigint an int
assert text(x, {"x": 2.0}) == "2" and value(x, {"x": "s"}) == "s"
assert text(undefined) == "undefined" and simplified(undefined).literal
assert text(name("NaN")) == "NaN" and text(name("Infinity")) == "Infinity"
assert text(undefined, {"undefined": 1.0}) == "1"  # a variable shadows a global
assert value(this, {"this": 1.5}) == 1.5 and text(this) == "this"  # `this`, bound by the scope by that name
assert value(null) is None and text(null) == "null" and repr(A.UNDEFINED) == "undefined"
# A name the scope does not bind stays, named in `free`; its root is no literal
unbound = simplified(binary("+", x, 1.0))
assert text(binary("+", x, 1.0)) == "x + 1" and unbound.free == {"x"} and unbound.residues == () and not unbound.literal
with raises(NotEvaluated, match="the root is not a literal: x is not bound"):
    unbound.evaluate()
# A value is written back as ECMAScript writes it: a negative number negated, NaN, Infinity and -0 by name, `undefined`
# by name, `null` and a regular expression a constant; each a literal root
for bound, written in [(-2.0, "-2"), (-0.0, "-0"), (math.inf, "Infinity"), (-math.inf, "-Infinity"), (math.nan, "NaN"),
                       (-3, "-3n"), (A.UNDEFINED, "undefined"), (None, "null"), ("a", '"a"'), (True, "true"),
                       (A.RegExp("a", ""), "/a/"), (A.RegExp("a+", "gu"), "/a+/gu")]:
    assert text(x, {"x": bound}) == written and simplified(x, {"x": bound}).literal
assert text(unary("-", 2.0)) == "-2" and text(regexp("a+", "g")) == "/a+/g"  # a regular expression, as written
assert A.type_of(value(regexp("a"))) == "object" and value(regexp("a+", "g")).flags == "g"  # a new object

<a id="TSS-02"></a>
## TSS-02 · Conversions: ToBoolean, ToNumber of strings and objects, ToString of numbers and objects, ToPrimitive by an object's own methods

In [ ]:
# ToBoolean, by `!`
for operand, negated in [(0.0, True), ("", True), ("0", False), (name("NaN"), True), (0, True), (null, True),
                         (undefined, True), (array(), False), (obj(), False), (arrow(x, x), False), (1, False)]:
    assert value(unary("!", operand)) is negated
# ToNumber, by `+`: StringToNumber, its white space trimmed; an object by its primitive
for operand, written in [("  12  ", "12"), ("", "0"), ("0x1F", "31"), ("0o17", "15"), ("0b101", "5"), ("-0x1", "NaN"),
                         ("1_000", "NaN"), ("Infinity", "Infinity"), ("-Infinity", "-Infinity"), ("infinity", "NaN"),
                         (".5", "0.5"), ("5.", "5"), ("1e3", "1000"), ("\u00a0\n7\ufeff", "7"), ("0x", "NaN"),
                         ("0x" + "f" * 300, "Infinity"), (null, "0"), (undefined, "NaN"), (True, "1"),
                         (array(), "0"), (array(5.0), "5"), (array(1.0, 2.0), "NaN"), (obj(), "NaN")]:
    assert text(unary("+", operand)) == written
with raises(TypeError, match="Cannot convert a BigInt value to a number"):
    value(unary("+", 1))
# ToString, by `"" + x`: Number::toString's shortest round trip, an array joined, an object's tag
for operand, written in [(1e21, "1e+21"), (1e-7, "1e-7"), (123.456, "123.456"), (binary("+", 0.1, 0.2), "0.30000000000000004"),
                         (unary("-", 0.0), "0"), (array(1.0, array(2.0, 3.0)), "1,2,3"),
                         (array(null, undefined, hole, 1.0), ",,,1"), (obj(), "[object Object]"),
                         (regexp("a", "g"), "/a/g"), (True, "true"), (False, "false"), (10, "10"), (null, "null"), (undefined, "undefined")]:
    assert value(binary("+", "", operand)) == written
# ToPrimitive: an object's own valueOf, then toString (toString first for a string), where they are functions
assert value(binary("+", obj(prop(name("valueOf"), arrow(5.0))), 1.0)) == 6.0
assert value(template("", obj(prop(name("toString"), arrow("s")), prop(name("valueOf"), arrow(1.0))), "")) == "s"
assert value(binary("+", obj(prop(name("valueOf"), arrow(obj())), prop(name("toString"), arrow("t"))), "")) == "t"
with raises(TypeError, match="Cannot convert object to primitive value"):
    value(binary("+", obj(prop(name("toString"), 1.0)), ""))
# A function's source text is not modeled: it stays, named
assert reasons(binary("+", arrow(x, x), "")) == [("TypeScript5 Function.prototype.toString is not simplified", "c")]

<a id="TSS-03"></a>
## TSS-03 · Numbers: IEEE 754 arithmetic, division by zero, remainder and exponentiation as ECMAScript defines them, and the 32-bit operators

In [ ]:
minus_zero = unary("-", 0.0)
for left, op, right, written in [
    (1.0, "+", 2.0, "3"), (0.1, "+", 0.2, "0.30000000000000004"), (7.0, "-", 10.0, "-3"), (2.0, "*", 3.0, "6"),
    (1.0, "/", 0.0, "Infinity"), (-1.0, "/", 0.0, "-Infinity"), (1.0, "/", minus_zero, "-Infinity"), (0.0, "/", 0.0, "NaN"),
    (name("NaN"), "/", 0.0, "NaN"), (5.0, "%", 3.0, "2"), (-5.0, "%", 3.0, "-2"), (5.0, "%", -3.0, "2"),
    (-4.0, "%", 2.0, "-0"), (5.0, "%", 0.0, "NaN"), (name("Infinity"), "%", 2.0, "NaN"), (5.0, "%", name("Infinity"), "5"),
    (0.0, "%", 5.0, "0"), (5.5, "%", 2.0, "1.5"), (2.0, "**", 10.0, "1024"), (2.0, "**", -1.0, "0.5"),
    (-8.0, "**", binary("/", 1.0, 3.0), "NaN"), (1.0, "**", name("Infinity"), "NaN"), (name("NaN"), "**", 0.0, "1"),
    (2.0, "**", name("NaN"), "NaN"), (name("NaN"), "**", 2.0, "NaN"), (0.0, "**", -1.0, "Infinity"),
    (minus_zero, "**", -1.0, "-Infinity"), (minus_zero, "**", 3.0, "-0"), (minus_zero, "**", 2.0, "0"),
    (0.0, "**", 2.0, "0"), (10.0, "**", 400.0, "Infinity"), (-10.0, "**", 401.0, "-Infinity"), (4.0, "**", 0.5, "2"),
    (-2.0, "**", 3.0, "-8"), (5.0, "&", 3.0, "1"), (5.0, "|", 3.0, "7"), (5.0, "^", 3.0, "6"),
    (1.0, "<<", 31.0, "-2147483648"), (1.0, "<<", 32.0, "1"), (-16.0, ">>", 2.0, "-4"), (-16.0, ">>>", 28.0, "15"),
    (-1.0, ">>>", 0.0, "4294967295"), (4294967301.0, "|", 0.0, "5"), (name("NaN"), "|", 0.0, "0"),
    (3.7, "|", 0.0, "3"), (-3.7, "|", 0.0, "-3"), (name("Infinity"), "|", 0.0, "0"),
    ("3", "*", "4", "12"), ("3", "-", 1.0, "2"), ("a", "*", 1.0, "NaN"), (True, "+", 1.0, "2"), (null, "+", 1.0, "1"),
    (undefined, "+", 1.0, "NaN"), (array(2.0), "*", array(3.0), "6")]:
    assert text(binary(op, left, right)) == written, (op, written)
for op, operand, written in [("~", 5.0, "-6"), ("-", "3", "-3"), ("-", null, "-0"), ("-", minus_zero, "0"),
                             ("~", "x", "-1"), ("-", array(), "-0")]:
    assert text(unary(op, operand)) == written

<a id="TSS-04"></a>
## TSS-04 · BigInt: arbitrary precision, truncating division, its errors, and its comparisons with numbers and strings

In [ ]:
for left, op, right, written in [
    (1, "+", 2, "3n"), (2, "**", 64, "18446744073709551616n"), (7, "/", 2, "3n"), (-7, "/", 2, "-3n"), (7, "%", -2, "1n"),
    (-7, "%", 2, "-1n"), (5, "-", 8, "-3n"), (3, "*", -4, "-12n"), (5, "&", 3, "1n"), (-5, "|", 2, "-5n"),
    (5, "^", -1, "-6n"), (1, "<<", 100, "1267650600228229401496703205376n"), (256, ">>", 4, "16n"), (-9, ">>", 1, "-5n"),
    (1, "<<", -1, "0n"), (8, ">>", -1, "16n"), (0, "**", 0, "1n"), (1, "**", 2 ** 40, "1n"), (0, "<<", 2 ** 40, "0n")]:
    assert text(binary(op, left, right)) == written, (op, written)
assert text(unary("-", 5)) == "-5n" and text(unary("~", 5)) == "-6n" and text(unary("-", unary("-", 5))) == "5n"
for spec, error, message in [
    (binary("+", 1, 1.0), TypeError, "Cannot mix BigInt and other types, use explicit conversions"),
    (binary("/", 1, 0), A.RangeError, "Division by zero"), (binary("%", 1, 0), A.RangeError, "Division by zero"),
    (binary("**", 2, -1), A.RangeError, "Exponent must be non-negative"),
    (binary(">>>", 1, 0), TypeError, "BigInts have no unsigned right shift, use >> instead"),
    (binary("**", 2, 2 ** 40), A.RangeError, "Maximum BigInt size exceeded"),
    (binary("<<", 1, 2 ** 31), A.RangeError, "Maximum BigInt size exceeded")]:
    with raises(error, match=message):
        value(spec)
# Compared with numbers mathematically, with strings by StringToBigInt (undefined where a string is no integer)
for left, op, right, result in [
    (1, "<", 2.0, True), (2, ">", "1", True), (1, "==", "1", True), (1, "==", "x", False), (1, "<", "x", False),
    ("x", ">", 1, False), ("1", "<", 2, True), ("x", "<", 1, False), (1, "==", 1.0, True), (1, "===", 1.0, False),
    (1, "==", 1.5, False), (1, "==", name("Infinity"), False), (name("NaN"), "<", 1, False), (2, ">", 1.5, True),
    ("0x10", "==", 16, True), (" 12 ", "==", 12, True), ("", "==", 0, True), ("1.0", "==", 1, False),
    ("-0x10", "==", -16, False), ("-5", "==", -5, True), (2 ** 64, ">", 1e19, True), (1.0, "==", 1, True),
    (1, "<", "2", True)]:
    assert value(binary(op, left, right)) is result, (left, op, right)

<a id="TSS-05"></a>
## TSS-05 · Equality and order: IsLooselyEqual, IsStrictlyEqual and IsLessThan, objects by identity, strings by UTF-16 code units

In [ ]:
same = arrow(binary("===", x, x), x)  # an object is itself
for left, op, right, result in [
    (null, "==", undefined, True), (null, "===", undefined, False), (null, "===", null, True), (null, "==", 0.0, False), (undefined, "==", 0.0, False),
    (name("NaN"), "==", name("NaN"), False), (name("NaN"), "===", name("NaN"), False), (0.0, "===", minus_zero, True),
    ("1", "==", 1.0, True), ("", "==", 0.0, True), ("a", "==", "a", True), (True, "==", 1.0, True), (True, "==", "1", True),
    (1.0, "==", False, False), (array(1.0), "==", 1.0, True), (array(1.0), "==", "1", True), (array(), "==", "", True),
    ("1", "==", array(1.0), True), (obj(), "==", "[object Object]", True), (array(), "==", array(), False),
    (obj(), "===", obj(), False), (null, "==", False, False), (1.0, "!=", "1", False), (1.0, "!==", "1", True),
    ("b", ">", "a", True), ("B", "<", "a", True), ("a", "<", "aa", True), ("\U00010000", "<", "\uffff", True),
    ("10", "<", "9", True), ("10", "<", 9.0, False), (1.0, "<", name("NaN"), False), (name("NaN"), ">=", 1.0, False),
    (1.0, "<=", 1.0, True), (2.0, ">=", 3.0, False), (null, ">=", 0.0, True), (undefined, ">=", 0.0, False),
    (array(2.0), ">", 1.0, True), (1.0, "<=", name("NaN"), False), (2.0, ">", 1.0, True)]:
    assert value(binary(op, left, right)) is result, (left, op, right)
assert value(call(same, array())) is True and value(call(same, obj())) is True

<a id="TSS-06"></a>
## TSS-06 · Strings and templates: UTF-16 code units, cooked and raw strings, and a tag called with them

In [ ]:
assert value(binary("+", "a", "b")) == "ab" and value(member("\U0001f600", name("length"))) == 2.0
assert value(member("\U0001f600", 0.0, computed=True)) == "\ud83d"  # a code unit, a lone surrogate
assert value(binary("+", "\ud83d", "\ude00")) == "\U0001f600" and value(binary("===", binary("+", "\ud83d", "\ude00"), "\U0001f600"))
assert text(array(spread("a\U0001f600"))) == '["a", "\U0001f600"]'  # a string's iterator gives code points
assert value(template("a", binary("+", 1.0, 1.0), "b")) == "a2b" and value(template("", array(1.0, 2.0), "")) == "1,2"
assert value(template("", null, "")) == "null" and text(template("a", x, "")) == "`a${x}`"
assert value(lambda u: u.templateliteral(lambda r: r.quasis([lambda q: q.tail(True)]))) == ""  # a quasi with no raw text
# A quasi's escapes cooked: single, hexadecimal, Unicode, NUL, any other character itself, a line continuation nothing;
# a line terminator read as LF
assert value(template(r"\n\t\x41\u0042\u{43}\0\'\`\$\\\q")) == "\n\tABC\0'`$\\q"
assert value(template("a\\\nb\r\nc\rd\\\u2028e")) == "ab\nc\nde"
for raw in [r"\1", r"\08", r"\x4", r"\u{110000}", r"\u12", "\\"]:
    with raises(ValueError, match="a templateliteral's quasi has an invalid escape"):
        value(template(raw))
# A tag is called with its strings, an array whose `raw` is their raw text, undefined where an escape is invalid, then
# the values between them
s, v = name("s"), name("v")
tag = arrow(array(member(member(s, name("raw")), 1.0, computed=True), member(s, 1.0, computed=True),
                  member(v, 0.0, computed=True), binary("in", "raw", s), member(s, name("length"))), s, rest(v))
assert text(tagged(tag, "a", 1.0, r"b\xZ")) == r'["b\\xZ", undefined, 1, true, 2]'
assert text(tagged(tag, "a", x, "b")) == "((s, ...v) => [s.raw[1], s[1], v[0], \"raw\" in s, s.length])`a${x}b`"
assert text(tagged(arrow(s, s), "a")) == "((s) => s)`a`"  # its strings have no display
with raises(TypeError, match="1 is not a function"):
    value(tagged(1.0, "a"))

<a id="TSS-07"></a>
## TSS-07 · Arrays and objects: holes, spreads and displays, keys in ECMAScript's order, member access, optional chains, `in`, data objects and host values

In [ ]:
# Arrays: holes, spreads by iteration; a display of literals, or the subterms simplified
assert text(array(1.0, hole, 2.0)) == "[1,, 2]" and text(array(spread(array(1.0, hole, 2.0)))) == "[1, undefined, 2]"
assert text(array(1.0, spread(x), binary("+", 2.0, 3.0))) == "[1, ...x, 5]" and text(array(hole, x)) == "[, x]"
assert text(array(spread(array(1.0, 2.0)), x)) == "[...[1, 2], x]" and text(array(arrow(x, x))) == "[(x) => x]"
# Objects: keys in ECMAScript's order, array indices first; computed keys; shorthands; spreads; a later key's value
assert text(obj(prop(name("b"), 1.0), prop(name("a"), 2.0), prop(1.0, 3.0), prop(0.0, 4.0))) == "({ 0: 4, 1: 3, b: 1, a: 2 })"
assert text(obj(prop(binary("+", "a", "b"), 1.0, computed=True), prop(x, x, shorthand=True)), {"x": 1.0}) == "({ ab: 1, x: 1 })"
assert text(obj(prop(1.5, 1.0), prop("a-b", null), prop(2, True))) == '({ 2: true, "1.5": 1, "a-b": null })'
assert text(obj(prop(name("a"), 1.0), prop(name("b"), 2.0), prop(name("a"), 3.0))) == "({ a: 3, b: 2 })"
assert text(obj(spread(obj(prop(name("a"), 1.0))), prop(name("b"), 2.0), spread(array(7.0)), spread("hi"), spread(null),
                spread(5.0))) == '({ 0: "h", 1: "i", a: 1, b: 2 })'
assert text(obj(prop(name("a"), binary("+", 1.0, 1.0)), prop(name("b"), x))) == "({ a: 2, b: x })"
assert text(obj(prop(name("a"), arrow(x, x)))) == "({ a: (x) => x })"  # a closure has no literal
assert text(obj(prop("__proto__", 1.0, computed=True))) == '({ ["__proto__"]: 1 })'  # a property, no prototype
assert reasons(obj(prop(name("__proto__"), null))) == [("TypeScript5 objectexpression __proto__ is not simplified", "a")]
# Member access: own properties, else undefined; a built-in method read but not modeled
for spec, written in [
    (member(array(1.0, 2.0), name("length")), "2"), (member(array(1.0, 2.0), 1.0, computed=True), "2"),
    (member(array(1.0, 2.0), 5.0, computed=True), "undefined"), (member(array(1.0, hole), 1.0, computed=True), "undefined"),
    (member(array(1.0, 2.0), "1", computed=True), "2"), (member(array(1.0), name("foo")), "undefined"),
    (member("abc", name("length")), "3"), (member("abc", 1.0, computed=True), '"b"'), (member("abc", 5.0, computed=True), "undefined"),
    (member(obj(prop(name("a"), 1.0)), name("a")), "1"), (member(obj(prop(name("a"), 1.0)), name("b")), "undefined"),
    (member(obj(prop(name("toString"), 1.0)), name("toString")), "1"), (member(1.0, name("foo")), "undefined"),
    (member(arrow(x, x), name("foo")), "undefined"), (member(regexp("a"), name("foo")), "undefined")]:
    assert text(spec) == written, written
for spec, builtin in [(member("abc", name("toUpperCase")), "String.prototype.toUpperCase"),
                      (member(array(), name("map")), "Array.prototype.map"),
                      (member(obj(), name("toString")), "Object.prototype.toString"),
                      (member(1.0, name("toFixed")), "Number.prototype.toFixed"),
                      (member(True, name("valueOf")), "Boolean.prototype.valueOf"),
                      (member(1, name("toString")), "BigInt.prototype.toString"),
                      (member(arrow(x, x), name("call")), "Function.prototype.call"),
                      (member(regexp("a"), name("test")), "RegExp.prototype.test"),
                      (member(array(), name("hasOwnProperty")), "Object.prototype.hasOwnProperty")]:
    assert reasons(spec) == [(f"TypeScript5 {builtin} is not simplified", "c")]
assert text(call(member("abc", name("toUpperCase")))) == '"abc".toUpperCase()'
for spec, message in [(member(null, name("a")), "Cannot read properties of null (reading 'a')"),
                      (member(undefined, 0.0, computed=True), "Cannot read properties of undefined (reading '0')"),
                      (chain(member(member(obj(prop(name("a"), null)), name("a"), optional=True), name("b"))),
                       "Cannot read properties of null (reading 'b')"),
                      (binary("in", 1.0, "abc"), "Cannot use 'in' operator to search for '1' in abc")]:
    with raises(TypeError, match=message):
        value(spec)
# An optional chain short-circuits to undefined where `?.` meets null or undefined
assert text(chain(member(null, name("a"), optional=True))) == "undefined"
assert text(chain(member(member(null, name("a"), optional=True), name("b")))) == "undefined"
assert text(chain(call(null, 1.0, optional=True))) == "undefined" and text(chain(member(x, name("a"), optional=True))) == "x?.a"
assert text(chain(call(arrow(1.0), optional=True))) == "1"
# `in`: own and inherited properties
for key, operand, result in [("length", array(), True), (0.0, array(1.0), True), (1.0, array(1.0, hole), False),
                             (2.0, array(1.0), False), ("map", array(), True), ("x", array(), False), ("raw", array(), False),
                             ("hasOwnProperty", obj(), True), ("a", obj(prop(name("a"), 1.0)), True), ("b", obj(), False),
                             ("length", arrow(x, x), True), ("foo", arrow(x, x), False), ("lastIndex", regexp("a"), True),
                             ("x", regexp("a"), False)]:
    assert value(binary("in", key, operand)) is result, (key, result)
# A data object's properties, as an object's: a list an array
with store.transaction():
    ann = store.tss.Person(lambda p: p.name("Ann").age(30).scores([1, 2]))
    nobody = store.tss.Person(lambda p: p)
p = name("p")
for spec, written in [(member(p, name("name")), '"Ann"'), (member(p, name("age")), "30n"), (member(p, name("scores")), "[1n, 2n]"),
                      (member(p, name("missing")), "undefined"), (binary("in", "name", p), "true"), (unary("typeof", p), '"object"'),
                      (obj(spread(p)), '({ name: "Ann", age: 30n, scores: [1n, 2n] })'), (binary("+", p, ""), '"[object Object]"'),
                      (binary("===", p, p), "true"), (binary("==", p, "[object Object]"), "true")]:
    assert text(spec, {"p": ann}) == written, written
assert value(binary("in", "age", p), {"p": nobody}) is False and text(p, {"p": ann}) == "p"  # an object has no literal
assert reasons(member(p, name("valueOf")), {"p": ann}) == [("TypeScript5 Object.prototype.valueOf is not simplified", "c")]
assert value(binary("+", obj(prop(name("valueOf"), arrow(1.0))), p), {"p": ann}) == "1[object Object]"
with store.transaction():  # its own `toString`, a string, no function: no primitive
    named = store.tss.Person(lambda p: p.toString("me"))
with raises(TypeError, match="Cannot convert object to primitive value"):
    value(binary("+", p, ""), {"p": named})
# A host value, any other the scope provides, has nothing ECMAScript reads: it stays, named
for spec in [member(x, name("a")), binary("+", x, ""), binary("in", "a", x), array(spread(x)), obj(spread(x))]:
    assert set(reasons(spec, {"x": object()})) == {("TypeScript5 host value is not simplified", "c")}
assert value(unary("typeof", x), {"x": object()}) == "object"

<a id="TSS-08"></a>
## TSS-08 · Arrows, calls and `new`: closures binding their parameters by destructuring, the scope's functions and classes, and what stays

In [ ]:
a, b, r = name("a"), name("b"), name("r")
for spec, written in [
    (call(arrow(binary("+", x, 1.0), x), 2.0), "3"), (call(arrow(binary("+", x, y), x, default(y, 10.0)), 1.0), "11"),
    (call(arrow(y, x, default(y, x)), 5.0), "5"), (call(arrow(y, x, default(y, 10.0)), 1.0, 2.0), "2"), (call(arrow(r, rest(r)), 1.0, 2.0), "[1, 2]"),
    (call(arrow(member(r, name("length")), a, rest(r)), 1.0, 2.0, 3.0), "2"), (call(arrow(b, a, b), 1.0), "undefined"),
    (call(arrow(array(a, b, c), array_pattern(a, hole, default(b, 9.0), rest(c))), array(1.0, 2.0, undefined, 4.0, 5.0)),
     "[1, 9, [4, 5]]"),
    (call(arrow(array(a, b), array_pattern(a, b)), array(1.0)), "[1, undefined]"),
    (call(arrow(array(a, c, r), object_pattern(prop(a, a, shorthand=True), prop(b, object_pattern(prop(c, c, shorthand=True))),
                                               rest(r))), obj(prop(a, 1.0), prop(b, obj(prop(c, 2.0))), prop(name("d"), 3.0))),
     "[1, 2, { d: 3 }]"),
    (call(arrow(a, object_pattern(prop(binary("+", "a", ""), a, computed=True))), obj(prop(a, 7.0))), "7"),
    (call(arrow(a, object_pattern(prop("a", a))), obj(prop(a, 8.0))), "8"),
    (call(arrow(binary("+", a, b), a, b), spread(array(1.0, 2.0))), "3"), (call(arrow(r, rest(r)), spread("ab")), '["a", "b"]'),
    (call(call(arrow(arrow(binary("+", x, y), y), x), 1.0), 2.0), "3"), (call(arrow(this)), "(() => this)()"),
]:
    assert text(spec) == written, written
assert value(call(arrow(this)), {"this": 4.0}) == 4.0  # `this`, an arrow's, its scope's
# A closure has no literal: it stays, its body simplified, its parameters unknown within it
assert text(arrow(x, x)) == "(x) => x" and text(arrow(binary("+", y, binary("+", 1.0, 2.0)), y)) == "(y) => y + 3"
assert text(arrow(binary("+", x, name("z")), x), {"z": 1.0}) == "(x) => x + 1"
assert text(call(arrow(binary("+", x, y), x), 2.0)) == "((x) => x + y)(2)"
for spec, message in [(call(arrow(a, object_pattern(prop(a, a, shorthand=True))), null), "Cannot destructure 'null' as it is null."),
                      (call(arrow(a, array_pattern(a)), 5.0), "number is not iterable"),
                      (call(arrow(a, array_pattern(a)), null), "null is not iterable"),
                      (array(spread(obj())), "object is not iterable"), (call(1.0, 2.0), "1 is not a function"),
                      (call(member(x, name("y"))), "x.y is not a function"), (new(arrow(x, x)), "(x) => x is not a constructor"),
                      (new(1.0), "1 is not a constructor")]:
    with raises(TypeError, match=message):
        value(spec, {"x": {"y": 1.0}})
with raises(ValueError, match="a memberexpression is not a parameter"):
    value(call(arrow(1.0, member(x, name("y"))), 1.0))
# A function and a class the scope provides are called, and an object made by one is an instance of it
class Point:
    def __init__(self, at):
        self.at = at


twice = lambda n: n * 2  # noqa: E731
scope = {"f": twice, "Point": Point}
assert value(call(f, 21.0), scope) == 42.0 and value(unary("typeof", f), scope) == "function" and text(f, scope) == "f"
assert value(binary("instanceof", new(name("Point"), 1.0), name("Point")), scope) is True
assert value(binary("instanceof", obj(), name("Point")), scope) is False and value(binary("instanceof", 1.0, f), scope) is False
assert value(unary("typeof", new(name("Point"), 1.0)), scope) == "object"
for spec, message in [(new(f), "f is not a constructor"),
                      (binary("instanceof", obj(), f), "Function has non-object prototype 'undefined' in instanceof check"),
                      (binary("instanceof", array(), arrow(x, x)), "Function has non-object prototype 'undefined' in instanceof check"),
                      (binary("instanceof", obj(), 1.0), "Right-hand side of 'instanceof' is not callable")]:
    with raises(TypeError, match=message):
        value(spec, scope)
# An async arrow's call and a block body's stay, named: a promise and statements are mbse-programs'
assert value(unary("typeof", arrow(x, x, is_async=True))) == "function"
assert reasons(call(arrow(x, x, is_async=True), 1.0)) == [("TypeScript5 async arrowfunctionexpression is not simplified", "d")]
assert reasons(call(arrow(None), 1.0)) == [("TypeScript5 arrowfunctionexpression block is not simplified", "d")]

<a id="TSS-09"></a>
## TSS-09 · Logical, conditional and sequence operators, short circuits around what stays, TypeScript's assertions erased, and what stays, by its class

In [ ]:
for spec, written in [
    (logical("&&", 1.0, 2.0), "2"), (logical("&&", 0.0, x), "0"), (logical("||", "", "d"), '"d"'), (logical("??", null, 5.0), "5"),
    (logical("??", 0.0, 5.0), "0"), (logical("&&", True, x), "x"), (logical("||", False, x), "x"), (logical("??", undefined, x), "x"),
    (logical("&&", x, 1.0), "x && 1"), (logical("||", 1.0, x), "1"), (logical("??", 1.0, x), "1"), (logical("||", x, True), "x || true"), (logical("||", arrow(x, x), y), "((x) => x) || y"),
    (conditional(1.0, "a", "b"), '"a"'), (conditional(c, 1.0, 2.0), "c ? 1 : 2"), (conditional(True, x, y), "x"),
    (conditional(0.0, x, y), "y"), (sequence(1.0, 2.0), "2"), (sequence(1.0, x), "x"), (sequence(x, 1.0), "x, 1"),
    (parenthesized(binary("+", 1.0, 2.0)), "3"), (parenthesized(x), "(x)"), (unary("void", 1.0), "undefined"),
    (unary("typeof", null), '"object"'), (unary("typeof", 1), '"bigint"'), (unary("typeof", undefined), '"undefined"'),
    (unary("typeof", x), "typeof x")]:
    assert text(spec) == written, written
# TypeScript's assertions are erased: their operand's value
for kind, arguments in [("tsasexpression", (x, number_type)), ("tssatisfiesexpression", (x, number_type)),
                        ("tstypeassertion", (number_type, x)), ("tsnonnullexpression", (x,)),
                        ("tsinstantiationexpression", None)]:
    spec = make(kind, {}, *arguments) if arguments else lambda u: u.tsinstantiationexpression(  # its helper's type
        lambda r: r.expression(x).typeArguments(lambda a: a.params([number_type])))
    assert value(spec, {"x": 1.5}) == 1.5
# A type a residual keeps is `TypeScript Types`', which simplification across a sub-dialect is to reach (CAP-299)
assert text(make("tsasexpression", {}, x, number_type)) == "x as number"
assert reasons(make("tsasexpression", {}, x, number_type)) == [("TypeScript Types tsnumberkeyword is not simplified", "b")]
# What stays, by its class: a write, an update, `delete`, `#x` (d); a signature without a body (b)
for spec, reason in [
    (make("assignmentexpression", {"operator": "="}, lambda p: p.identifier(lambda i: i.name("x")), 1.0), ("TypeScript5 assignmentexpression is not simplified", "d")),
    (make("updateexpression", {"operator": "++", "prefix": True}, x), ("TypeScript5 updateexpression is not simplified", "d")),
    (unary("delete", member(x, name("a"))), ("TypeScript5 unaryexpression 'delete' is not simplified", "d")),
    (member(this, make("privateidentifier", {"name": "x"})), ("TypeScript5 privateidentifier is not simplified", "d")),
    (make("tsemptybodyfunctionexpression", {"isAsync": False, "generator": False}, None, None), ("TypeScript5 tsemptybodyfunctionexpression is not simplified", "b"))]:
    assert reasons(spec, {"this": obj()})[0] == reason, reason
assert TV.FOLDED == {"literal", "constant", "identifier", "thisexpression", "templateliteral", "taggedtemplateexpression",
                     "parenthesizedexpression", "sequenceexpression", "elision", "spreadelement", "arrayexpression",
                     "property", "objectexpression", "memberexpression", "chainexpression", "callexpression",
                     "newexpression", "arrowfunctionexpression", "unaryexpression", "binaryexpression",
                     "logicalexpression", "conditionalexpression", "tsasexpression", "tssatisfiesexpression",
                     "tstypeassertion", "tsnonnullexpression", "tsinstantiationexpression"}
assert TV.SIMPLIFIER.category("TypeScript5", "String.prototype.at", "memberexpression") == "c" and len(A.BUILTINS) == 139